In [ ]:
import random
random.seed(42)

# Modell 444 LSTM with Keras

### Was kann dieses Modell?

### Daten: vier Rennen
### Modelle: LSTM
### Training: sequentiell
### Vorhersage: Probs für alle Fahrer
### Problem: ???

## Imports

In [1]:
import fastf1
from fastf1 import plotting
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# sklearn related imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import brier_score_loss
from sklearn.utils.class_weight import compute_sample_weight



project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)
from utils.Feature_Engineer import FeatureEngineer
from utils.Metrics import Metrics
from utils.Models import Models
from utils.experiment_db import init_db
init_db()


pd.set_option('display.width', None)         # passt die Ausgabe-Breite an
pd.set_option('display.max_columns', None)   # zeigt alle Spalten
pd.set_option('display.max_rows', None)      # zeigt alle Reihen

%load_ext autoreload
%autoreload 2

## Rennen definieren

In [2]:
schedule = fastf1.get_event_schedule(2024, include_testing=False)
print(schedule["EventName"].tolist())

req         WARNING 	DEFAULT CACHE ENABLED! (2.5 GB) /Users/lukaskoch/Library/Caches/fastf1


['Bahrain Grand Prix', 'Saudi Arabian Grand Prix', 'Australian Grand Prix', 'Japanese Grand Prix', 'Chinese Grand Prix', 'Miami Grand Prix', 'Emilia Romagna Grand Prix', 'Monaco Grand Prix', 'Canadian Grand Prix', 'Spanish Grand Prix', 'Austrian Grand Prix', 'British Grand Prix', 'Hungarian Grand Prix', 'Belgian Grand Prix', 'Dutch Grand Prix', 'Italian Grand Prix', 'Azerbaijan Grand Prix', 'Singapore Grand Prix', 'United States Grand Prix', 'Mexico City Grand Prix', 'São Paulo Grand Prix', 'Las Vegas Grand Prix', 'Qatar Grand Prix', 'Abu Dhabi Grand Prix']


In [3]:
YEAR = 2024
GRAND_PRIX = ['Austrian Grand Prix', 'British Grand Prix', 'Hungarian Grand Prix', 'Belgian Grand Prix']
SESSION_TYPE = "R"

In [4]:
all_laps = []
for gp in GRAND_PRIX:
    try:
        session = fastf1.get_session(YEAR, gp, SESSION_TYPE)
        session.load(weather=True, laps=True, telemetry=True)
        laps = session.laps.copy()
        weather = session.weather_data.copy()
        
        laps = laps.sort_values("Time")
        weather = weather.sort_values("Time")

        merged = pd.merge_asof(
                laps,
                weather,
                on="Time",
                direction="backward"
            )
        
        merged['GP'] = gp
        merged['Year'] = YEAR
        
        all_laps.append(merged)
        print(f"✔ Loaded {gp}")
        
    except Exception as e:
        print(f"Error loading {gp}: {e}")
        
data = pd.concat(all_laps, ignore_index=True)

core           INFO 	Loading data for Austrian Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['63', '81', '55', '44', '1', '27', '11', '20', '3', '10', '16', '31', '18', '22', '23', '77', '24', '14', '2', '4']
core           INFO 	Loading data for British Grand Prix - Ra

✔ Loaded Austrian Grand Prix


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['44', '1', '4', '81', '55', '27', '18', '14', '23', '22', '2', '20', '3', '16', '77', '31', '11', '24', '63', '10']
core           INFO 	Loading data for Hungarian Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


✔ Loaded British Grand Prix


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['81', '4', '44', '16', '1', '55', '11', '63', '22', '18', '14', '3', '27', '23', '20', '77', '2', '31', '24', '10']
core           INFO 	Loading data for Belgian Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Fixed incorrect tyre stint information fo

✔ Loaded Hungarian Grand Prix


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['44', '81', '16', '1', '4', '55', '11', '14', '31', '3', '18', '23', '10', '20', '77', '22', '2', '27', '24', '63']


✔ Loaded Belgian Grand Prix


In [5]:
fe = FeatureEngineer()
merged = fe.create_features(data)

In [6]:
features = fe.choose_features(merged)
features

['gapAheadSeconds',
 's1_delta',
 's2_delta',
 's3_delta',
 'lap_pace_delta',
 'is_raining',
 'position_change',
 'compound_HARD',
 'compound_INTERMEDIATE',
 'compound_MEDIUM',
 'compound_SOFT',
 'status_1',
 'status_12',
 'status_16',
 'status_167',
 'status_21',
 'status_671',
 'status_71']

In [7]:
race_order = (
    merged.groupby(["GP", "Year"])["LapStartDate"]
      .min()
      .sort_values()
      .reset_index()[["GP", "Year"]]
)

race_order

,GP,Year
0,Austrian Grand Prix,2024
1,British Grand Prix,2024
2,Hungarian Grand Prix,2024
3,Belgian Grand Prix,2024


In [8]:
train_races = race_order.iloc[:3]
test_race = race_order.iloc[-1:]

train_df = merged.merge(train_races, on=["GP", "Year"]).sort_values(["GP", "Year", "LapNumber", "Driver"])
test_df = merged.merge(test_race, on=["GP", "Year"]).sort_values(["GP", "Year", "LapNumber", "Driver"])

train_df[features] = train_df[features].astype('float32')
test_df[features]  = test_df[features].astype('float32')

In [9]:
X_train_raw = train_df[features].values
y_train_raw = train_df['pit_in_next_lap'].values

X_test_raw = test_df[features].values
y_test_raw = test_df['pit_in_next_lap'].values

In [10]:
SEQ_LEN = 10

def create_sequences(features, labels, seq_len):
    X = []
    y = []
    for i in range(len(features) - seq_len):
        X.append(features[i:i + seq_len])
        y.append(labels[i + seq_len])
    return np.array(X), np.array(y) 


X_train, y_train = create_sequences(X_train_raw, y_train_raw, SEQ_LEN)
X_test, y_test = create_sequences(X_test_raw, y_test_raw, SEQ_LEN)

In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam

In [12]:
from tensorflow.keras.metrics import AUC, Precision, Recall

model = Sequential([
    
    LSTM(64, input_shape=(SEQ_LEN, len(features))),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=Adam(1e-3),
    loss='binary_crossentropy',
    metrics=[
        AUC(name="auc"), 
        Precision(name="precision"), 
        Recall(name="recall")]
)


model.summary()

/Users/lukaskoch/anaconda3/envs/base_3_12/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        21,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,313 (83.25 KB)

 Trainable params: 21,313 (83.25 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(zip(np.unique(y_train), class_weights))
print(class_weights)

{0: 0.5182960893854749, 1: 14.164122137404581}


In [14]:
history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.1,
    class_weight= class_weights
)

Epoch 1/100
105/105 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - auc: 0.5850 - loss: 0.7048 - precision: 0.0439 - recall: 0.6875 - val_auc: 0.4268 - val_loss: 0.8083 - val_precision: 0.0083 - val_recall: 1.0000
Epoch 2/100
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - auc: 0.6699 - loss: 0.6651 - precision: 0.0567 - recall: 0.6953 - val_auc: 0.4783 - val_loss: 0.6391 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 3/100
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - auc: 0.7130 - loss: 0.6383 - precision: 0.0618 - recall: 0.6172 - val_auc: 0.5226 - val_loss: 0.5111 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 4/100
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - auc: 0.7375 - loss: 0.6183 - precision: 0.0725 - recall: 0.6484 - val_auc: 0.5316 - val_loss: 0.5209 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 5/100
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - auc: 0.7659 - loss: 0.5955 - precision: 0.0778 - recall: 0.7109 - val_auc: 0.4684 - val_loss: 0.4437 - val_precision:

In [15]:
y_pred_prob = model.predict(X_test)

26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


In [16]:
from utils.experiment_db import log_or_update_metrics

metrics = Metrics.calculate_metrics(f"approach_444_LSTM", y_test, y_pred_prob)
Metrics.print_metrics(metrics)
    
    
log_or_update_metrics(
    model_name=f"approach_444_{LSTM}",
    metrics={
        **metrics
    }
)

   ✔ Brier-Score:              0.0714 -> optimal 0.0
   ✔ Log-Loss:                 0.5711 -> optimal 0.0
   ✔ ROC-AUC:                  0.4614 -> optimal 1.0
   ✔ PR-AUC:                   0.0442 -> optimal 1.0
